# Topic: A/B Testing

## Definition (30-second explanation)
* A/B Testing (or Split Testing) is the gold standard of controlled experiments in the industry, used to make data-driven product decisions.
* It involves randomly assigning users to two groups: the Control group (A) sees the original version, and the Treatment group (B) sees the new version.
* You then compare a key metric between the two groups to determine whether the change caused a statistically significant improvement.

## Why Interviewers Ask This
* Virtually every major tech company uses A/B testing for product decisions.
* Interviewers want to ensure you follow rigorous experimental design (like pre-determining sample size) rather than making amateur mistakes like peeking.
* To verify you can connect statistical results to real business impacts, such as converting a percentage lift into additional revenue.

## Core Concepts
* **Randomization:** Users must be randomly assigned to avoid selection bias.
* **Lift:** The improvement in the metric. **Absolute Lift** is the raw difference ($p_B - p_A$), while **Relative Lift** is the percentage improvement ($(p_B - p_A) / p_A$).
* **Sample Ratio Mismatch (SRM):** A critical data quality check to ensure the actual traffic split matches the designed split (e.g., exactly 50/50).
* **Peeking:** Stopping an experiment early just because the p-value temporarily dips below 0.05, which drastically inflates the Type I error (false positives).

## When to Use
* Testing UI designs (button colors, layouts), email subjects, or ad creatives.
* Evaluating new machine learning recommendation algorithms against existing ones.
* Validating new pricing structures, discount offers, or any product change before a full 100% rollout.

## Advantages
* Establishes clear causality between a product change and user behavior (unlike observational data).
* Bayesian A/B testing alternatives allow for continuous monitoring and dynamic traffic allocation without the strict fixed-sample-size rules of frequentist tests.

## Limitations
* Should not be used when you cannot randomly assign users (e.g., geographic constraints).
* Fails when there are **network effects** (where changing one user's experience affects another); requires cluster randomization instead.
* Impractical if you have very few users (e.g., <100 per group) or if the experiment would take too long to collect enough data.

## Common Comparisons
* **Absolute Lift vs. Relative Lift:** Absolute lift shows the raw point difference, while relative lift contextualizes the improvement against the baseline. Both, along with a confidence interval, should be reported.
* **Frequentist vs. Bayesian:** Frequentist requires a fixed sample size determined by power analysis, while Bayesian computes the probability that B is better than A using Beta distributions.
* **A/B Testing vs. Multi-Armed Bandits:** A/B tests wait until the end to declare a winner; bandits dynamically allocate more traffic to the winning variant *during* the test to minimize lost conversions.

## Common Interview Traps
* **TRAP:** Peeking at results and stopping early. Always pre-determine test duration and stick to it.
* **TRAP:** Testing too many metrics without pre-specifying a primary metric, which increases the chance of finding a false positive.
* **TRAP:** Ignoring Sample Ratio Mismatch (SRM). If one group unexpectedly gets more traffic, the entire test is invalid.

## Python / SQL Syntax 
    import numpy as np
    from statsmodels.stats.proportion import proportions_ztest

    # Run two-proportion z-test
    counts = np.array([conv_B, conv_A])
    nobs = np.array([n_B, n_A])
    z_stat, p_value = proportions_ztest(counts, nobs)
    
    # Calculate lifts
    absolute_lift = p_B - p_A
    relative_lift = (p_B - p_A) / p_A

## Important Formula 
* **Pooled Proportion:** $p = \frac{\text{conversions}_A + \text{conversions}_B}{n_A + n_B}$
* **Z-Statistic:** $z = \frac{p_B - p_A}{\sqrt{p(1-p)(1/n_A + 1/n_B)}}$

## 45-Second Interview Answer
"A/B testing is a randomized controlled experiment used to determine the causal impact of a product change. To run one correctly, you must first pre-determine your required sample size and duration using a power analysis. Once the test completes, you check for data quality issues like Sample Ratio Mismatch before calculating the z-statistic and p-value. If significant, you report both the absolute and relative lift, and most importantly, translate that statistical lift into tangible business impact like projected annualized revenue."

## Practice Questions:

### Q1:
**Question:** What is Sample Ratio Mismatch (SRM)? How do you detect it?

**Answer:**
Sample Ratio Mismatch (SRM) occurs when the actual traffic split between the Control and Treatment groups deviates significantly from the designed split (e.g., observing a 52/48 split when you configured a 50/50 split). It indicates a severe flaw in the randomization engine, logging pipeline, or experimental setup, rendering the test results invalid. To detect it, you run a Chi-Square Goodness-of-Fit test on the observed traffic counts versus the expected traffic counts. If the p-value is very low (often < 0.001 for SRM checks), you have an SRM and must discard the experiment.

**Common Mistakes Candidates Make:**
* Assuming a slight imbalance (e.g., 5005 vs 4995 users) is definitely an SRM without running a statistical test to check if it's within expected random variance.
* Proceeding to analyze the conversion metrics without checking for SRM first.

**Interviewer Follow-up:**
"If you detect an SRM, what are some common engineering or product reasons that could have caused it?" (Answer: SRM is usually caused by bugs in the randomization hashing function, bot traffic disproportionately hitting one variant, or the treatment variant loading too slowly and causing users to drop off before being logged.)

### Q2:
**Question:** How would you run an A/B test when users can be in both groups (e.g., logged-out users)?

**Answer:**
When users are logged out, we cannot use a persistent User ID for randomization. Instead, we must use a robust, persistent device identifier or a long-lasting browser cookie (like a session ID or device ID). The hashing function will map this cookie to a variant. However, this introduces the risk of identity fragmentation—if a user clears their cookies or switches from mobile to desktop, they might get bucketed into both Control and Treatment, ruining the user experience and violating the independence assumption. We must track cross-device behavior where possible and consider excluding users who were exposed to multiple variants from the final analysis.

**Common Mistakes Candidates Make:**
* Suggesting IP address for randomization (which changes frequently and groups entire households/offices together).
* Failing to mention the violation of the independence assumption if a user sees both variants.

**Interviewer Follow-up:**
"How would you clean the data post-experiment if you realize 5% of your traffic saw both the Control and Treatment experiences?" (Answer: You must entirely exclude that 5% of cross-contaminated users from your analysis, as their exposure to both variants violates the independent observations assumption.)

### Q3:
**Question:** You run 5 A/B tests at the same time on the same users. What is the problem?

**Answer:**
The primary problem is **interaction effects**. If multiple teams are testing changes on the same page or user flow simultaneously (e.g., Team A changes the button color, Team B changes the price layout), the combined effect might artificially inflate or depress the overall conversion rate. You won't be able to isolate which test caused the impact. To solve this, you need to use an experimental layering architecture (like multi-variate testing or orthogonal layers) to ensure the tests are mutually exclusive, or intentionally design a fractional factorial experiment to measure the interaction effects explicitly.

**Common Mistakes Candidates Make:**
* Confusing this with the "multiple testing problem" (which is testing 5 *metrics* in one test, rather than 5 *different tests* overlapping).
* Suggesting we just "never run concurrent tests," which is impossible for large companies that run thousands of tests a year.

**Interviewer Follow-up:**
"How do companies like Uber or Netflix structure their experimentation platforms to allow hundreds of concurrent tests without them colliding?" (Answer: They use orthogonal experimental layers and unique hashing salts to ensure that a user's random assignment in one test is completely independent of their assignment in overlapping tests.)

### Q4:
**Question:** Your A/B test shows p=0.04 after 3 days but you planned for 2 weeks. What do you do?

**Answer:**
I would do absolutely nothing except let the test continue running for the full 2 weeks. Stopping an experiment early because it crossed the significance threshold is a classic mistake known as "peeking." Because p-values fluctuate randomly over time, peeking dramatically inflates the Type I error rate (false positives). Furthermore, 3 days does not capture full weekly seasonality (e.g., weekend vs. weekday behavior). You must always stick to the pre-determined duration calculated during your initial power analysis.

**Common Mistakes Candidates Make:**
* Conceding to business pressure and saying, "I would stop the test and ship it to get the revenue faster."
* Not mentioning the loss of weekly seasonality data by stopping at 3 days.

**Interviewer Follow-up:**
"If the Product Manager says the new feature is causing the app to crash for 10% of users, does your answer change?" (Answer: Yes, you must immediately halt the test; "guardrail metrics" like crash rates or severe revenue drops always override the pre-determined test duration to protect the business.)